# 1. Environment and MDP Formalization

In [1]:
import gymnasium as gym
import PyFlyt.gym_envs
import numpy as np

### Probe environment

In [2]:
def probe_env(env_name, kwargs=None, steps=150, n_episodes=3):
    """
    Probes an environment and computes episode return statistics.
    """

    if kwargs is None:
        kwargs = {}

    print(f"\n=== {env_name} ===")

    env = gym.make(env_name, **kwargs)

    print("Observation space:", env.observation_space)
    print("Action space:", env.action_space)

    episode_returns = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=42 + ep)

        rewards = []

        for _ in range(steps):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, _ = env.step(action)

            rewards.append(reward)

            if terminated or truncated:
                break

        episode_returns.append(sum(rewards))

    print("\nEpisode return stats:")
    print(" mean:", np.mean(episode_returns))
    print(" std :", np.std(episode_returns))

    env.close()

In [3]:
modes = [-1, 0, 4, 6, 7]    

Hover

In [ ]:
hover_env = "PyFlyt/QuadX-Hover-v4"

for mode in modes:
    print(mode)
    probe_env(
        hover_env,
        kwargs={"flight_mode": mode},
        steps=150
    )


=== PyFlyt/QuadX-Hover-v4 ===
Observation space: Box(-inf, inf, (21,), float64)
Action space: Box(0.0, 0.8, (4,), float64)
                                                          
                             

Episode return stats:
 mean: -174.12495271655862
 std : 28.136313049717984

=== PyFlyt/QuadX-Hover-v4 ===
Observation space: Box(-inf, inf, (21,), float64)
Action space: Box([-3.14159265 -3.14159265 -3.14159265  0.        ], [3.14159265 3.14159265 3.14159265 0.8       ], (4,), float64)
                                                          
                             

Episode return stats:
 mean: -84.89585787259584
 std : 7.546555733506429

=== PyFlyt/QuadX-Hover-v4 ===
Observation space: Box(-inf, inf, (21,), float64)
Action space: Box([-3.14159265 -3.14159265 -3.14159265  0.        ], [3.14159265 3.14159265 3.14159265 0.8       ], (4,), float64)
                                                                                       

Episode return stats:
 mean: 52.344

Waypoints

In [5]:
waypoints_env = "PyFlyt/QuadX-Waypoints-v4"

for mode in modes:
    probe_env(
        waypoints_env,
        kwargs={
            "flight_mode": mode,
            "goal_reach_distance": 4.0,
            "flight_dome_size": 150.0,
            "max_duration_seconds": 120.0,
            "num_targets": 4
        },
        steps=150
    )


=== PyFlyt/QuadX-Waypoints-v4 ===
Observation space: Dict('attitude': Box(-inf, inf, (21,), float64), 'target_deltas': Sequence(Box(-300.0, 300.0, (3,), float64), stack=True))
Action space: Box(0.0, 0.8, (4,), float64)
                             
                                                          

Episode return stats:
 mean: -133.00385803057927
 std : 7.52519435083238

=== PyFlyt/QuadX-Waypoints-v4 ===
Observation space: Dict('attitude': Box(-inf, inf, (21,), float64), 'target_deltas': Sequence(Box(-300.0, 300.0, (3,), float64), stack=True))
Action space: Box([-3.14159265 -3.14159265 -3.14159265  0.        ], [3.14159265 3.14159265 3.14159265 0.8       ], (4,), float64)
                                                                                       

Episode return stats:
 mean: 59.63714608692634
 std : 71.69018050859188

=== PyFlyt/QuadX-Waypoints-v4 ===
Observation space: Dict('attitude': Box(-inf, inf, (21,), float64), 'target_deltas': Sequence(Box(-300.0, 300.0, 

Flattening

In [6]:
env = gym.make(
    "PyFlyt/QuadX-Waypoints-v4",
    flight_mode=6,
    goal_reach_distance=4.0,
    flight_dome_size=150.0,
    max_duration_seconds=120.0,
    num_targets=4
)

obs, _ = env.reset(seed=42)

print("Observation type:", type(obs))

if isinstance(obs, dict):
    total_dim = 0

    for k, v in obs.items():
        print(k, v.shape)
        total_dim += int(np.prod(v.shape))

    print("\nFlattened dimension:", total_dim)

env.close()

                             
Observation type: <class 'dict'>
attitude (21,)
target_deltas (4, 3)

Flattened dimension: 33


Reward

In [7]:
env = gym.make("PyFlyt/QuadX-Hover-v4", flight_mode=0)

all_rewards = []

for ep in range(5):
    obs, _ = env.reset(seed=42 + ep)

    for _ in range(300):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, _ = env.step(action)

        all_rewards.append(reward)

        if terminated or truncated:
            break

env.close()

all_rewards = np.array(all_rewards)

print("Reward mean:", all_rewards.mean())
print("Reward std :", all_rewards.std())
print("Reward min :", all_rewards.min())
print("Reward max :", all_rewards.max())

                             
                                                                                                                    
Reward mean: -3.816398218237303
Reward std : 20.429239902131965
Reward min : -101.5791013009087
Reward max : 2.7005720708329575


Transitions

In [8]:
env = gym.make("PyFlyt/QuadX-Hover-v4", flight_mode=0)
obs, _ = env.reset(seed=42)

deltas = []

for _ in range(50):
    action = env.action_space.sample()
    next_obs, _, _, _, _ = env.step(action)

    deltas.append(np.linalg.norm(next_obs - obs))
    obs = next_obs

env.close()

print("Mean state change:", np.mean(deltas))
print("Std state change :", np.std(deltas))

                             
Mean state change: 2.9029025738511387
Std state change : 3.3175871176258274


Dogfight

In [10]:
from scripts.dogfight_wrapper import DogfightSelfPlayEnv

env = DogfightSelfPlayEnv()

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

episode_returns = []

for ep in range(5):
    obs, _ = env.reset()

    rewards = []

    for _ in range(200):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, _ = env.step(action)

        rewards.append(reward)

        if terminated or truncated:
            break

    episode_returns.append(sum(rewards))

env.close()

print("Return mean:", np.mean(episode_returns))
print("Return std :", np.std(episode_returns))

Observation space: Box(-inf, inf, (37,), float64)
Action space: Box(-1.0, 1.0, (4,), float64)
                                                                                                                                                 
Return mean: 1320.1068006515502
Return std : 49.905223517532


# MDP Formalization summary

1. QuadX-Hover
    - S: $\mathbb R^{21}$ (continuous state vector)
    - P: deterministic physics-based dynamics (PyBullet), effectively deterministic given state-action
    - R: dense stabilisation reward

2. QuadX-Waypoints
    - S: $\mathbb R^{21}$ (concatenation of drone state and fixed-size padded waypoint deltas)
    - P: physics + waypoint logic
    - R: sparse + shaped navigation reward

3. Dogfight (self-play MDP approximation)
    - S: $\mathbb R^{37}$ (aircraft + opponent state + relative and/or extra features)
    - P: stochastic (opponent policy + physics)
    - R: sparse combat reward (hit/survival)

Action space: $\mathbb R^4$

Difficulty respective domains:
- $-1$: $[ 0.0, 0.8 ]$
- $0, 4, 6, 7$: $[ (-\pi, -\pi, -\pi, 0), (\pi, \pi, \pi, 0.8)]$


All environments are episodic MDPs.